# Notebook 01 — Document Processing

**Goal:** Ingest engineering PDFs → extract text → chunk → embed → store in ChromaDB.

**Steps:**
1. Verify PDF corpus in `data/documents/`
2. Test `src/ingestion.py` extraction on a single file
3. Experiment with chunk sizes: 300 / 500 / 800 tokens
4. Ingest all documents into ChromaDB
5. Smoke-test retrieval: 3 sample queries, inspect returned chunks

**Output:** Populated ChromaDB collection, `data/eval/ingested_hashes.json`, chunk count per doc.

In [ ]:
# Standard imports
import os, json, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd()))
print('Working dir:', Path.cwd())

In [ ]:
# ── Verify PDF corpus ──────────────────────────────────────────────────
from pathlib import Path
doc_dir = Path('data/documents')
pdfs = sorted(doc_dir.glob('*.pdf'))
print(f'Found {len(pdfs)} PDFs:')
for p in pdfs:
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:50s}  {size_kb:,.0f} KB')

# TODO: Add engineering PDFs to data/documents/ before running this
# Sources listed in PROJECT_BRIEF.md → Dataset section

In [ ]:
# ── Test extraction on a single PDF ────────────────────────────────────
from src.ingestion import extract_pages, chunk_pages

if not pdfs:
    print('No PDFs found. Add files to data/documents/ first.')
else:
    sample_pdf = pdfs[0]
    print(f'Extracting: {sample_pdf.name}')
    pages = extract_pages(sample_pdf)
    print(f'  Pages extracted: {len(pages)}')
    print(f'  First page preview (500 chars):')
    print(pages[0]['text'][:500])

In [ ]:
# ── Chunk size experiment: compare 300 / 500 / 800 ────────────────────
if pdfs:
    for chunk_size in [300, 500, 800]:
        chunks = chunk_pages(pages, chunk_size=chunk_size, overlap=chunk_size // 10)
        avg_chars = sum(len(c['text']) for c in chunks) / len(chunks) if chunks else 0
        print(f'chunk_size={chunk_size}: {len(chunks):4d} chunks, avg_chars={avg_chars:.0f}')

# Observation: too small → noisy retrieval (clause split mid-sentence)
# too large → answer buried in noise; 500 tok typically best for standards docs

In [ ]:
# ── Full ingestion ──────────────────────────────────────────────────────
from src.ingestion import ingest

# First run: --reset to start fresh. Subsequent runs are idempotent (hash-diff).
result = ingest(reset=False, chunk_size=500, overlap=50)
print(result)

In [ ]:
# ── Smoke-test retrieval ────────────────────────────────────────────────
from src.retriever import Retriever

retriever = Retriever(use_hybrid=True, use_reranker=False)  # no reranker yet

test_queries = [
    'minimum pipe insulation thickness for steam pipe ASHRAE',
    'hydrostatic test Class 900 flanges ASME',
    'machine guarding requirements OSHA',
]
for q in test_queries:
    chunks = retriever.retrieve(q, top_k=3)
    print(f'Q: {q}')
    for i, c in enumerate(chunks, 1):
        print(f'  [{i}] {c["source"]} p.{c["page"]}  score={c.get("rrf_score", c.get("score", 0)):.3f}')
    print()

## Summary

| Metric | Value |
|--------|-------|
| PDFs ingested | *(fill in)* |
| Total chunks | *(fill in)* |
| Avg chunk size (chars) | *(fill in)* |
| Smoke test: correct source in top-3 | *(fill in)* / 3 |

**Best chunk size:** 500 tokens with 50-token overlap — *(confirm after testing)*

**Next:** Notebook 02 — Retrieval Evaluation + Ablations